# Replicate the system

This notebook runs the whole pipeline once, exactly as `./flight --execute_flight --offline` does, and then
walks through what each stage produced. It calls the flight itself, so nothing here can drift from the
scripts.

| Stage | What runs |
|---|---|
| 1 install | the `.venv` environment and the pinned CLIP model (skipped here with `--skip-install`) |
| 2 collect | Reddit and Mapillary through their APIs (skipped here: `--offline`) |
| 3 load models | CLIP ViT-B/32 and VADER, each smoke-tested |
| 4 results | every published output rebuilt from the committed raw data; tests; verification checks; figures; report numbers; the report; the notebooks |
| 5 checks | every rebuilt file compared with its published copy; nothing in `Data/`, `Docs/`, `Deliverables/` or `Scripts/` changed; no credential in the output |

The run goes into `Notebooks/_out/flight/EXECUTION-<n>/` (git-ignored) and takes about two minutes.
For a full run with the API calls, use `./flight --execute_flight` from the repository root.

In [ ]:
# Shared folders, project modules and plot settings
from _setup import *
# Show plots inside the notebook
%matplotlib inline
# Running the flight as a separate program, and reading its JSON status file
import json
import subprocess

## Step 1: Run the flight

In [ ]:
# Where this notebook's runs go (git-ignored)
runs = out_dir("flight")
# The flight, offline and without reinstalling packages, writing into that folder
command = [str(REPO / "flight"), "--execute_flight", "--offline", "--skip-install", "--results-root", str(runs)]
# Run it from the repository root and keep what it prints
result = subprocess.run(command, cwd=REPO, capture_output=True, text=True, check=False)
# The last lines say whether the run passed and where its summary is
print("\n".join(result.stdout.splitlines()[-3:]))

## Step 2: Every step and its status

PASS = worked as designed; WARN = worked, with a known limitation; FAIL = did not work; SKIP = not run.

In [ ]:
# The newest run folder (EXECUTION-<n> with the largest n)
latest = max(runs.glob("EXECUTION-*"), key=lambda p: int(p.name.split("-")[1]))
# The run's status file: one entry per step
status = json.loads((latest / "flight_status.json").read_text())
# One row per step: its number, stage, name, status and result
steps = pd.DataFrame(status["steps"])[["n", "stage", "id", "status", "note"]]
steps

In [ ]:
# How many steps of each status in each stage
steps.pivot_table(index="stage", columns="status", values="id", aggfunc="count", fill_value=0)

## Step 3: Rebuilt files against the published copies

identical = the same bytes; equivalent = differs only where a run stamps its own time (the workbook's creation time, report headers).

In [ ]:
# Each rebuilt file with how it compares to its published copy
compare = pd.read_csv(latest / "comparisons" / "reproduction_vs_published.csv")
# Files per area and result
compare.pivot_table(index="area", columns="status", values="file", aggfunc="count", fill_value=0)

## Step 4: The workflow figure from this run

In [ ]:
# The workflow and status figure the run drew
picture = plt.imread(latest / "figures" / "fig01_workflow_status.png")
# Show it without axes
fig, ax = plt.subplots(figsize=(10, 7))
ax.imshow(picture)
ax.axis("off")
plt.show()

## Takeaway

- One command rebuilds every published output from the committed raw data and compares each file with
  its published copy.
- The full summary of this run is `FLIGHT_SUMMARY.md` in the run folder; every step's log is in `logs/`.